Lấy tất cả bài viết trong ngày

In [10]:
import requests
from bs4 import BeautifulSoup
from datetime import date
import time
import random
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datetime import datetime, timedelta

# 1. Cấu hình

In [11]:

headers = {"User-Agent": "Mozilla/5.0"}
ngay = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
all_links = []
page = 1

# 2. Lấy danh sách link

In [12]:
print(f"=== LINK NGÀY {ngay} ===")
while True:
    if page == 1:
        url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}.htm"
    else:
        url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}/trang-{page}.htm"

    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    articles = soup.find_all(
        "article",
        class_="article-item",
        attrs={"data-content-name": f"category-timeline_page_{page}"}
    )

    if not articles:
        break

    for art in articles:
        link = art.get("data-content-target")
        if link:
            if link.startswith("/"):
                link = "https://dantri.com.vn" + link
                all_links.append(link)
    page += 1
print("\nTỔNG LINK:", len(all_links))
for l in all_links:
    print(l)


=== LINK NGÀY 2026-02-07 ===

TỔNG LINK: 45
https://dantri.com.vn/thoi-su/ha-noi-mua-ret-dam-nhiet-do-thap-nhat-10-12-do-c-20260207215802601.htm
https://dantri.com.vn/thoi-su/dong-nai-sap-hoan-tat-thao-do-khu-cong-nghiep-bien-hoa-1-20260206170904909.htm
https://dantri.com.vn/tet/cu-mai-150-tuoi-xuat-hien-tai-le-hoi-xuan-can-gio-20260207214744226.htm
https://dantri.com.vn/thoi-su/nu-giao-vien-tu-vong-trong-tai-nan-o-tphcm-20260207204702246.htm
https://dantri.com.vn/thoi-su/tay-ninh-tang-cuong-kiem-tra-nong-do-con-dip-tet-binh-ngo-20260207193022182.htm
https://dantri.com.vn/thoi-su/6-o-to-dam-nhau-tren-cau-nhat-tan-giao-thong-te-liet-20260207204552858.htm
https://dantri.com.vn/thoi-su/22-uy-vien-bo-chinh-tri-ban-bi-thu-se-dan-doan-kiem-tra-40-to-chuc-dang-20260207183825839.htm
https://dantri.com.vn/thoi-su/dong-nai-lap-to-cong-tac-ho-tro-giai-phong-mat-bang-du-an-72000-ty-dong-20260207171753147.htm
https://dantri.com.vn/thoi-su/tau-ca-mac-can-o-cua-bien-chu-tau-cau-cuu-bo-doi-bien-phong-

# 3. Hàm crawl

In [13]:
def crawl_article(url):
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    # ===== TITLE =====
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    # ===== DATE =====
    # Tìm thẻ <time> có thuộc tính datetime, bất kể class là gì
    time_tag = soup.find("time", attrs={"datetime": True})
    if time_tag:
        # Ưu tiên lấy giá trị trong thuộc tính datetime (thường chuẩn hóa hơn)
        published_date = time_tag.get("datetime")
        # Nếu muốn lấy text hiển thị (ví dụ: "Thứ bảy, 24/01/2026 - 11:45") thì dùng:
        # published_date = time_tag.get_text(strip=True)
    else:
        published_date = ""

    # ===== BODY =====
    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if text:
            paragraphs.append(text)

    body = "\n".join(paragraphs)

    return {
        "url": url,
        "title": title,
        "date": published_date,
        "content": body
    }

In [14]:
print("\n=== TEST 1 BÀI ===")
if all_links:
    test_url = all_links[0] # Lấy bài đầu tiên để test
    data = crawl_article(test_url)

    print("URL:", data["url"])
    print("TITLE:", data["title"])
    print("DATE:", data["date"])
    print("CONTENT:")
    print(data["content"][:500] + "...") # In 500 ký tự đầu
else:
    print("Không tìm thấy link nào để test.")



=== TEST 1 BÀI ===
URL: https://dantri.com.vn/thoi-su/ha-noi-mua-ret-dam-nhiet-do-thap-nhat-10-12-do-c-20260207215802601.htm
TITLE: Hà Nội mưa, rét đậm, nhiệt độ thấp nhất 10-12 độ C
DATE: 2026-02-08 00:00
CONTENT:
Trung tâm Dự báo khí tượng thủy văn quốc gia cho biết, ngày 7/2 không khí lạnh đã ảnh hưởng đến khu vực Đông Bắc Bắc Bộ gây mưa rải rác, nhiệt độ giảm 2-4 độ C.
Dự báo ngày 8/2, không khí lạnh được tăng cường mạnh hơn, ảnh hưởng đến các nơi khác ở Đông Bắc Bộ, sau đó lan sang Tây Bắc Bộ, Bắc Trung Bộ và Trung Trung Bộ.
Trên đất liền gió đông bắc cấp 3, vùng ven biển cấp 3-4.
Ngày 8 đến sáng sớm 9/2, Bắc Bộ và Bắc Trung Bộ có mưa, mưa rào, có nơi có dông; trời rét đậm, vùng núi rét hại.
Từ 8/2 Bắ...


# 4. Cấu hình Splitter

In [15]:
if 'data' in locals() and data:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        separators=[
            "\n\n",   # đoạn
            "\n",     # dòng
            ".", "!", "?", ":", ";", ","
        ]
    )
    dulieu = splitter.split_text(data["content"])
    print("Số chunk:", len(dulieu))
    for i, chunk in enumerate(dulieu):
        print(f"\n--- CHUNK {i} ---")
        print(chunk)


Số chunk: 7

--- CHUNK 0 ---
Trung tâm Dự báo khí tượng thủy văn quốc gia cho biết, ngày 7/2 không khí lạnh đã ảnh hưởng đến khu vực Đông Bắc Bắc Bộ gây mưa rải rác, nhiệt độ giảm 2-4 độ C.
Dự báo ngày 8/2, không khí lạnh được tăng cường mạnh hơn, ảnh hưởng đến các nơi khác ở Đông Bắc Bộ, sau đó lan sang Tây Bắc Bộ, Bắc Trung Bộ và Trung Trung Bộ.
Trên đất liền gió đông bắc cấp 3, vùng ven biển cấp 3-4.
Ngày 8 đến sáng sớm 9/2, Bắc Bộ và Bắc Trung Bộ có mưa, mưa rào, có nơi có dông; trời rét đậm, vùng núi rét hại.
Từ 8/2 Bắc Trung Bộ trời rét, có nơi rét đậm. Nhiệt độ thấp nhất đợt này ở Bắc Bộ phổ biến 10-13 độ C, vùng núi 7-10 độ C, vùng núi cao có nơi dưới 3 độ C. Bắc Trung Bộ phổ biến 12-15 độ C.
Khu vực Hà Nội ngày 8/2 có mưa, mưa rào; ngày 8-9/2 trời rét đậm, nhiệt độ thấp nhất 10-12 độ C.

--- CHUNK 1 ---
Khu vực Hà Nội ngày 8/2 có mưa, mưa rào; ngày 8-9/2 trời rét đậm, nhiệt độ thấp nhất 10-12 độ C.
Do ảnh hưởng của không khí lạnh khu vực Bắc Biển Đông (bao gồm đặc khu Hoàng Sa

# 5. Xử lý hàng loạt

In [16]:
final_data = []
print("\n=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===")

for i, link in enumerate(all_links):
    print(f"[{i+1}/{len(all_links)}] Đang xử lý: {link}")

    article_data = crawl_article(link)

    if article_data and article_data["content"]:
        # Split text
        chunks = splitter.split_text(article_data["content"])

        # Lưu kết quả
        final_data.append({
            "url": link,
            "title": article_data["title"],
            "date": article_data["date"],
            "chunks": chunks
        })
        print(f"  -> OK: {len(chunks)} chunks | Date: {article_data['date']}")
    else:
        print("  -> Bỏ qua (không lấy được nội dung)")

    # Rate limiting
    sleep_time = random.uniform(1, 2)
    time.sleep(sleep_time)

print(f"\n=== HOÀN THÀNH ===")
print(f"Đã xử lý thành công: {len(final_data)} bài.")



=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===
[1/45] Đang xử lý: https://dantri.com.vn/thoi-su/ha-noi-mua-ret-dam-nhiet-do-thap-nhat-10-12-do-c-20260207215802601.htm
  -> OK: 7 chunks | Date: 2026-02-08 00:00
[2/45] Đang xử lý: https://dantri.com.vn/thoi-su/dong-nai-sap-hoan-tat-thao-do-khu-cong-nghiep-bien-hoa-1-20260206170904909.htm
  -> OK: 3 chunks | Date: 2026-02-07 23:06
[3/45] Đang xử lý: https://dantri.com.vn/tet/cu-mai-150-tuoi-xuat-hien-tai-le-hoi-xuan-can-gio-20260207214744226.htm
  -> OK: 4 chunks | Date: 2026-02-07 23:01
[4/45] Đang xử lý: https://dantri.com.vn/thoi-su/nu-giao-vien-tu-vong-trong-tai-nan-o-tphcm-20260207204702246.htm
  -> OK: 2 chunks | Date: 2026-02-07 21:21
[5/45] Đang xử lý: https://dantri.com.vn/thoi-su/tay-ninh-tang-cuong-kiem-tra-nong-do-con-dip-tet-binh-ngo-20260207193022182.htm
  -> OK: 2 chunks | Date: 2026-02-07 21:15
[6/45] Đang xử lý: https://dantri.com.vn/thoi-su/6-o-to-dam-nhau-tren-cau-nhat-tan-giao-thong-te-liet-20260207204552858.htm
  -> OK: 2 c

In [17]:
# 6. Lưu file JSON
output_file = "dulieu.json"
result = {
    "date": ngay,
    "articles": final_data
} 
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)
print(f"Đã lưu dữ liệu vào file: {output_file}")

Đã lưu dữ liệu vào file: dulieu.json
